`docker run --gpus '"device=2"' -it -p 309:8888 -v ~${DATA_MOUNT}:${DATA_MOUNT} -v ~/mnt/deepcell:${SPARK_ACQUISITION_MOUNT} -v ~${DATA_MOUNT2}:${DATA_MOUNT2} changhua/pymc5-jupyter:latest`

In [1]:
!pip uninstall -y h5py
!pip install --no-cache-dir h5py
!pip install netCDF4

Found existing installation: h5py 3.14.0
Uninstalling h5py-3.14.0:
  Successfully uninstalled h5py-3.14.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 119.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 89.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 68.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [netCDF4]


In [1]:
import pandas as pd
import numpy as np
from tqdm import tqdm
import pymc as pm
import pytensor.tensor as pt
import arviz as az

In [14]:
screen_df = pd.read_csv('./decoupling_out/raw_screen_count_mapped.csv').iloc[:,1:]

In [15]:
screen_df

,barcode_seq,pre_1,mtx_1,pre_2,mtx_2,kinase_name,substrate_aa,paabd
0,ATGGATAATCCACAAAGGG,100,5,143,16,TNK1_HUMAN,KWKKDGVYLNLVIDE,sh2
1,AAAGTGTCGATCGACTCAG,107,4,97,17,TNK1_HUMAN,RLMTGDTYTAHAGAK,sh2
2,ACCTGGCAACCGGGGGTTG,133,6,141,20,TNK1_HUMAN,VDPGERLYDLNMPAY,sh2
3,ATGTCCAGCCGGGAAGAAG,122,4,120,23,FRK_HUMAN,DKFYKYVYSVLDDVI,sh2
4,ACGTACCCGTTGCAATTGG,185,5,138,37,TNK1_HUMAN,RGVCAERYSQEVFEK,sh2
...,...,...,...,...,...,...,...,...
262967,ACTGCGATACCGGGACCGG,583,11716,2039,85552,KKCC1_HUMAN,GSRQRSSSDPPAVHP,ww
262968,ACAAATACATGACCTCCCG,92,1175,751,39131,KS6B1_HUMAN_active,PRIEEEKSEEETSAP,ww
262969,AGGCCGTTAGCGCTCTCCG,350,12067,1052,63497,PLK1_HUMAN_active,KVTTVVATPGQGPDR,ww
262970,AGTGTCGTGCCGGTACCGG,774,48946,830,56183,PHKG1_HUMAN,CQMSLFASPTSSEEQ,ww


In [16]:
import numpy as np
import pandas as pd

def prepare_long_table_for_pymc(
    screen_df: pd.DataFrame,
    *,
    paabd_value: str,
    barcode_col: str = "barcode_seq",
    kinase_col: str = "kinase_name",
    substrate_col: str = "substrate_aa",
    paabd_col: str = "paabd",
    pre_cols: tuple[str, str] = ("pre_1", "pre_2"),
    mtx_cols: tuple[str, str] = ("mtx_1", "mtx_2"),
    pseudocount: float = 1.0,
):
    """
    Returns a long table with one row per (substrate, kinase, rep):
      columns: substrate, kinase, rep, pre, mtx, sf_pre, sf_mtx, log_offset
    where size factors sf_* are computed on (paabd, condition, rep) across ALL kinases/substrates.
    """
    df = screen_df[screen_df[paabd_col] == paabd_value].copy()

    # numeric + sum barcodes to substrate×kinase level (per replicate column)
    for c in list(pre_cols) + list(mtx_cols):
        df[c] = pd.to_numeric(df[c], errors="coerce")
    df = df.dropna(subset=list(pre_cols) + list(mtx_cols))

    grp = [kinase_col, substrate_col]
    agg = {c: "sum" for c in list(pre_cols) + list(mtx_cols)}
    df_sum = df.groupby(grp, as_index=False).agg(agg)

    # build long per rep
    rows = []
    for r, (pre_c, mtx_c) in enumerate(zip(pre_cols, mtx_cols), start=1):
        tmp = df_sum[[kinase_col, substrate_col, pre_c, mtx_c]].copy()
        tmp = tmp.rename(columns={kinase_col: "kinase", substrate_col: "substrate", pre_c: "pre", mtx_c: "mtx"})
        tmp["rep"] = r
        rows.append(tmp)

    long = pd.concat(rows, ignore_index=True)

    # size factors on (paabd, condition, rep): totals across all kinases/substrates
    # Here we compute two totals per rep: total_pre_r and total_mtx_r
    sf_rows = []
    for r, (pre_c, mtx_c) in enumerate(zip(pre_cols, mtx_cols), start=1):
        total_pre = float(df_sum[pre_c].sum())
        total_mtx = float(df_sum[mtx_c].sum())
        sf_rows.append({"rep": r, "sf_pre": total_pre, "sf_mtx": total_mtx})
    sf = pd.DataFrame(sf_rows)

    # median-scale (optional but usually nice numerically)
    sf["sf_pre"] = sf["sf_pre"] / (np.median(sf["sf_pre"]) if np.median(sf["sf_pre"]) > 0 else 1.0)
    sf["sf_mtx"] = sf["sf_mtx"] / (np.median(sf["sf_mtx"]) if np.median(sf["sf_mtx"]) > 0 else 1.0)

    long = long.merge(sf, on="rep", how="left")

    # offsets for raw-count likelihood:
    # log(mu_raw) = log(sf_mtx) + log(pre/sf_pre + c) + ...
    long["log_offset"] = np.log(long["sf_mtx"].clip(1e-12)) + np.log((long["pre"] / long["sf_pre"]) + pseudocount)

    # keep as integers for NB
    long["pre"] = np.rint(long["pre"]).astype(int)
    long["mtx"] = np.rint(long["mtx"]).astype(int)
    return long

In [17]:
def fit_decoupled_nb_pymc(
    long: pd.DataFrame,
    *,
    draws: int = 1000,
    tune: int = 1000,
    chains: int = 4,
    target_accept: float = 0.9,
    seed: int = 0,
):
    """
    long must have columns: substrate, kinase, rep, mtx, log_offset
    Fits hierarchical NB:
      log(mu) = log_offset + g_k + delta_{k,rep} + b_{substrate,kinase_centered}
    """
    # index coding
    substrates = pd.Index(sorted(long["substrate"].unique()))
    kinases = pd.Index(sorted(long["kinase"].unique()))
    reps = pd.Index(sorted(long["rep"].unique()))

    sub_idx = substrates.get_indexer(long["substrate"])
    kin_idx = kinases.get_indexer(long["kinase"])
    rep_idx = reps.get_indexer(long["rep"])

    y = long["mtx"].values.astype(int)
    log_offset = long["log_offset"].values.astype(float)

    coords = {
        "obs": np.arange(len(long)),
        "substrate": substrates.astype(str).tolist(),
        "kinase": kinases.astype(str).tolist(),
        "rep": reps.astype(int).tolist(),
    }

    with pm.Model(coords=coords) as model:
        # data containers
        sub_i = pm.Data("sub_i", sub_idx, dims="obs")
        kin_i = pm.Data("kin_i", kin_idx, dims="obs")
        rep_i = pm.Data("rep_i", rep_idx, dims="obs")
        log_off = pm.Data("log_off", log_offset, dims="obs")

        # ---- kinase-global shared across substrates ----
        sigma_g = pm.HalfNormal("sigma_g", 1.0)
        g = pm.Normal("g", 0.0, sigma=sigma_g, dims="kinase")  # kinase-global

        # ---- replicate-pair (block) deviations per kinase ----
        sigma_delta = pm.HalfNormal("sigma_delta", 0.5)
        delta = pm.Normal("delta", 0.0, sigma=sigma_delta, dims=("kinase", "rep"))
        
        # ---- binding term per substrate×kinase, centered per kinase ----
        sigma_b = pm.HalfNormal("sigma_b", 1.0)
        b_raw = pm.Normal("b_raw", 0.0, sigma=sigma_b, dims=("substrate", "kinase"))

        # center per kinase (mean across substrates = 0 for each kinase)
        b_centered = pm.Deterministic(
            "b",
            b_raw - pt.mean(b_raw, axis=0, keepdims=True),
            dims=("substrate", "kinase"),
        )

        # ---- NB dispersion ----
        alpha = pm.HalfNormal("alpha", 1.0)  # NB2 overdispersion

        # linear predictor for each observation
        eta = (
            log_off
            + g[kin_i]
            + delta[kin_i, rep_i]
            + b_centered[sub_i, kin_i]
        )

        mu = pm.Deterministic("mu", pt.exp(eta), dims="obs")

        # likelihood
        pm.NegativeBinomial("y", mu=mu, alpha=alpha, observed=y, dims="obs")

        idata = pm.sample(
            draws=draws,
            tune=tune,
            chains=chains,
            target_accept=target_accept,
            random_seed=seed,
        )

    return model, idata

In [18]:
def summarize_decoupled_effects(idata, long: pd.DataFrame):
    # posterior means
    g_mean = idata.posterior["g"].mean(dim=("chain","draw")).to_pandas()
    delta_mean = idata.posterior["delta"].mean(dim=("chain","draw")).to_pandas()
    b_mean = idata.posterior["b"].mean(dim=("chain","draw")).to_pandas()

    # average delta over reps -> one kinase-level adjustment
    delta_bar = delta_mean.mean(axis=1)  # index=kinase

    # build tidy table for (substrate, kinase)
    out = (
        b_mean.stack().rename("binding_strength").reset_index()
        .rename(columns={"level_0":"substrate", "level_1":"kinase"})
    )
    out["kinase_global"] = out["kinase"].map(g_mean) + out["kinase"].map(delta_bar)
    out["pair_total_effect"] = out["kinase_global"] + out["binding_strength"]

    return out


In [19]:
paabd_ls = ['sh2','fha','1433','ww']

In [20]:
for paabd in paabd_ls:
    sim_id = f'251125_pymc_{paabd}'
    print(f"processing {sim_id}")
    long = prepare_long_table_for_pymc(screen_df, paabd_value=paabd, pseudocount=1.0)
    model, idata = fit_decoupled_nb_pymc(long, draws=1000, tune=1000, chains=4, target_accept=0.9, seed=1)
    az.to_netcdf(idata,'./decoupling_out/' + sim_id + '.nc')
    print(f'saved {paabd}.nc')
    effects_df = summarize_decoupled_effects(idata, long)
    effects_df.to_csv('./decoupling_out/' + sim_id + '.csv')
    print(f'saved {paabd}.csv')


processing 251125_pymc_sh2


/opt/conda/envs/pytorch_env/lib/python3.9/site-packages/pymc/data.py:434: UserWarning: The `mutable` kwarg was not specified. Before v4.1.0 it defaulted to `pm.Data(mutable=True)`, which is equivalent to using `pm.MutableData()`. In v4.1.0 the default changed to `pm.Data(mutable=False)`, equivalent to `pm.ConstantData`. Use `pm.ConstantData`/`pm.MutableData` or pass `pm.Data(..., mutable=False/True)` to avoid this warning.
  warnings.warn(
Auto-assigning NUTS sampler...
Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [sigma_g, g, sigma_delta, delta, sigma_b, b_raw, alpha]


Sampling 4 chains for 1_000 tune and 1_000 draw iterations (4_000 + 4_000 draws total) took 5526 seconds.


saved sh2.nc
saved sh2.csv
processing 251125_pymc_fha


/opt/conda/envs/pytorch_env/lib/python3.9/site-packages/pymc/data.py:434: UserWarning: The `mutable` kwarg was not specified. Before v4.1.0 it defaulted to `pm.Data(mutable=True)`, which is equivalent to using `pm.MutableData()`. In v4.1.0 the default changed to `pm.Data(mutable=False)`, equivalent to `pm.ConstantData`. Use `pm.ConstantData`/`pm.MutableData` or pass `pm.Data(..., mutable=False/True)` to avoid this warning.
  warnings.warn(
Auto-assigning NUTS sampler...
Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [sigma_g, g, sigma_delta, delta, sigma_b, b_raw, alpha]


...

Sampling 4 chains for 1_000 tune and 1_000 draw iterations (4_000 + 4_000 draws total) took 3694 seconds.
The rhat statistic is larger than 1.01 for some parameters. This indicates problems during sampling. See https://arxiv.org/abs/1903.08008 for details


saved 1433.nc
saved 1433.csv
processing 251125_pymc_ww


/opt/conda/envs/pytorch_env/lib/python3.9/site-packages/pymc/data.py:434: UserWarning: The `mutable` kwarg was not specified. Before v4.1.0 it defaulted to `pm.Data(mutable=True)`, which is equivalent to using `pm.MutableData()`. In v4.1.0 the default changed to `pm.Data(mutable=False)`, equivalent to `pm.ConstantData`. Use `pm.ConstantData`/`pm.MutableData` or pass `pm.Data(..., mutable=False/True)` to avoid this warning.
  warnings.warn(
Auto-assigning NUTS sampler...
Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [sigma_g, g, sigma_delta, delta, sigma_b, b_raw, alpha]


Sampling 4 chains for 1_000 tune and 1_000 draw iterations (4_000 + 4_000 draws total) took 3846 seconds.
The rhat statistic is larger than 1.01 for some parameters. This indicates problems during sampling. See https://arxiv.org/abs/1903.08008 for details


saved ww.nc
saved ww.csv


In [79]:
from __future__ import annotations

from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr
import arviz as az


def summarize_idata_netcdf_significance(
    nc_path: str | Path,
    *,
    var_names: tuple[str, ...] = ("g", "b", "delta"),
    hdi_prob: float = 0.94,
) -> pd.DataFrame:
    nc_path = Path(nc_path)
    idata = az.from_netcdf(nc_path)
    if not hasattr(idata, "posterior"):
        raise ValueError(f"No posterior group found in {nc_path}")
    
    post = idata.posterior
    missing = [v for v in var_names if v not in post.data_vars]
    if missing:
        raise KeyError(f"Variables not found in posterior: {missing}. Available: {list(post.data_vars)}")
    for v in var_names:
        da = post[v]  # dims: chain, draw, ...
    
        mean_da = da.mean(dim=("chain", "draw"))
        sd_da = da.std(dim=("chain", "draw"))
    
        prob_pos_da = (da > 0).mean(dim=("chain", "draw"))
        lfsr_da = xr.apply_ufunc(lambda p: np.minimum(p, 1.0 - p), prob_pos_da)
        p2_da = xr.apply_ufunc(lambda x: np.minimum(1.0, 2.0 * x), lfsr_da)
    
        # HDI
        hdi = az.hdi(da, hdi_prob=hdi_prob)
    
        # arviZ uses a dim named "hdi" with coordinate values like ["lower","higher"]
        if "hdi" not in hdi.dims:
            raise ValueError(f"Unexpected HDI output dims for {v}: {hdi.dims}")
    
        hdi_low = hdi.sel(hdi="lower")
        hdi_high = hdi.sel(hdi="higher")
        stats = xr.Dataset(
        {
            "mean": mean_da,
            "sd": sd_da,
            "prob_pos": prob_pos_da,
            "lfsr": lfsr_da,
            "p_two_sided": p2_da,
        }
        )
        stats.to_dataframe().reset_index()
        hdi_low = hdi_low.rename({'b':'hdi_low'})
        hdi_high = hdi_high.rename({'b':'hdi_high'})
        stats = stats.merge(hdi_low,compat='override')
        stats = stats.merge(hdi_high,compat='override')
        df = stats.to_dataframe().reset_index()
        df.insert(0, "var", v)
        df = df.loc[:,['kinase','substrate','mean','sd','lfsr']]
        return df


In [82]:
for paabd in paabd_ls:
    print(paabd)
    nc_path = f"./decoupling_out/251125_pymc_{paabd}.nc"
    stats_df = summarize_idata_netcdf_significance(nc_path, var_names=("b"))
    stats_df.to_csv(f'./decoupling_out/251125_pymc_binding_stats_{paabd}.csv' )

sh2
fha
1433
ww
